### 49. Group Anagrams
Given an array of strings `strs`, group the anagrams together. You can return the answer in **any order**.

**Example 1:**
```shell
Input: strs = ["eat","tea","tan","ate","nat","bat"]
Output: [["bat"],["nat","tan"],["ate","eat","tea"]]
```
**Explanation:**

- There is no string in `strs` that can be rearranged to form `"bat"`.
- The strings `"nat"` and `"tan"` are anagrams as they can be rearranged to form each other.
- The strings `"ate"`, `"eat"`, and `"tea"` are anagrams as they can be rearranged to form each other.

**Example 2:**
```shell
Input: strs = [""]
Output: [[""]]
```
**Example 3:**
```shell
Input: strs = ["a"]
Output: [["a"]]
```
**Constraints:**

- `1 <= strs.length <= 10^4`
- `0 <= strs[i].length <= 100`
- `strs[i]` consists of lowercase English letters.

**Try alone before next session:** It's the natural extension of today's Valid Anagram: instead of checking if two strings are anagrams, you need a way to use frequency (or sorted-string) as a hash map _key_ to group many strings at once. Bring your Step 1–4 reasoning next time before touching code.

### Reasoning
**Step 1 - restate the problem yourself**

- I have a list of strings that should be grouped in separate lists if they are anagrams each other and added to the general resulting list. If a string doesn't have anagram it is left alone in the list.
- **Input**: a list of strings
- **Output**: a list of list of strings
- What are bounds of input size?
- Can the input be empty?
- Are there duplicate strings?
- Is the output order important?

1. **Bounds**: up to `10⁴` strings in the list; each individual string can be `0` to `100` characters long — notice that means individual strings **can be empty**.
2. **Empty input**: no, `strs` itself has at least 1 element (`[[]]`).
3. **Duplicate strings**: yes, possible — two identical strings are trivially anagrams of each other (same letters, same frequencies), so they belong in the same group.
4. **Output order**: doesn't matter — problem explicitly says return the groups in any order.

**Step 2**: the problem statement already gave you three examples, including two edge cases baked in. Trace the main example (`["eat","tea","tan","ate","nat","bat"]`) enough to state what determines whether two strings land in the same group. Then specifically address: what should happen with the empty-string case (`[""]`) given your grouping logic?
- Two strings land in the same group if they are anagrams. If given an empty string if should land in its own group, and treated as a valid group in the resulting list.

Good on the general grouping criterion and correctly handling `[""]`. Quick refinement: what if the input had **two** empty strings, e.g. `strs = ["", ""]`? Given your own answer earlier that identical strings count as anagrams of each other, would they land in the same group or separate ones?
- Yes, two empty strings will land in the same group (list).

**Step 3 — brute force approach**: how would you naively check every string against every other string to find its anagram group, and what's the time complexity?

**Compare each string vs group members to find the right group or create a new one**

In [60]:
def group_anagrams_brute(strs: list[str]) -> list[list[str]]:
    groups: list[list[str]] = []

    # string loop (n * n * m = n^2 * m = O(n^2 * m))
    for string in strs:   # O(n)
        # groups loop
        for group in groups:  # O(n) - groups can be up to n
            # 1. member check 1. len not match - skip to the next group
            if len(string) != len(group[0]): continue

            # 2. char comparison loop (m-character comparison inside the n² loop)
            for char in string:  # O(m) - m is longest string
                # member check 2
                if string.count(char) != group[0].count(char):
                    # not match - skip to the next group
                    break  # go to next group
            else:
                # happy path. match - add to the current group
                group.append(string)
                break  # go to next group
        else:
            # 3. new group created
            groups.append([string])
    # 4.
    return groups

In [62]:
group_anagrams_brute(["ab", "abc", "eat","tea","tan","ate","nat","bat"])
# group_anagrams_brute([""])
# group_anagrams_brute(["a"])

[['ab'], ['abc'], ['eat', 'tea', 'ate'], ['tan', 'nat'], ['bat']]

#### Algorithm
For `n` incoming strings do comparison against **each** of the existing group's member (up to `n` groups if no anagrams at all), char by char using `count()`.
If the length of the incoming string and the group's member unequal - skip the group, this is the wrong one.

If the length is confirmed, the strings char comparison starts:
- **no match** - stop comparison - `break` **the char comparison loop** for the current group member
- **match** - **the char comparison loop** completes and runs its `else` clause:
    - add to the current group (anagram found - the happy path)
    - `break` **the groups loop** - switch to the next incoming string. (is crucial, not let to run **the groups loop's** `else` clause, where the new group is created with the current string)
A new group is only created in the `else` block of **the groups loop**, after every existing group has been checked and none matched.

The length-check issue in particular is the second time today that exact failure mode has shown up (Valid Anagram, then here) — good sign that it's becoming a reflex:
> **whenever you're comparing character-count structures**, ask "did I confirm the lengths match, or could one side have extra characters the check never looks at?"

[Visual](../images/42Group_anagrams_brute.png)

#### Time & Space complexity

- Time complexity: $O(n^2 * m)$ since the `m`-character comparison happens inside the `n²` loop, once for every (string, group) pair, not as a separate one-time cost. That's **multiplicative**, not additive: $O(n² · m)$.
- Space complexity: $O(n)$ since the input is only rearranged in the groups and stored.

**Step 4 - select algorithm + state complexity**

**The sorted-string-as-key (Sorting + hash map)**. 

In [ ]:
def group_anagrams(strs: list[str]) -> list[list[str]]:
    groups = {}  # e.g. {"aet": ["eat", "tea", "ate"]}

    for string in strs:
        sorted_string = "".join(sorted(string))
        if sorted_string in groups:
            groups[sorted_string].append(string)
        else:
            groups[sorted_string] = [string]

    return list(groups.values())

In [ ]:
group_anagrams(["eat","tea","tan","ate","nat","bat"])

[['eat', 'tea', 'ate'], ['tan', 'nat'], ['bat']]

#### Algorithm
Use a hash map and the **sorted version** of the incoming strings as keys mapping them to the list of anagrams, groups.
1. Get the sorted version of the input string - used as the key.
2. **Lookup** by this key in the hash map for the correct anagram group:
    - found - append the **original string** (not sorted)
    - otherwise, add new key (sorted string) mapping to the newly created group with the current **original string** (not sorted)
3. return groups in a list

[Visual](../images/42Group_anagrams_hashmap_sort.png)

This solution eliminates redundant work of scanning up to `n` groups 

**Step 9 - complexity analysis**

- Time complexity: $O(n \times m\space log\space m)$ where `n` - number of strings, `m` - length of the longest string. **Sort every one of the n strings**, each costing $O(m\space log\space m)$.
- Space complexity: $O(n \times m)$ for the values, + $O(k \times m)$ for the keys (`k ≤ n` distinct sorted-signatures), giving $O(n \times m)$ overall — same form as the time complexity, which makes sense since you touch every character of every string at least once either way.

A single string of length `m` isn't free to store; it takes `m` units of memory, same as an array of `m` characters would.

So, if you have `n=10,000` strings, each up to `m=100` characters, and every one of those strings ends up stored somewhere in your `res` dict's value lists (each string lands in exactly one group) — is the total memory closer to `10,000` units, or `10,000 × 100 = 1,000,000` units? What does counting just "k keys + n strings" as if they were single units miss?
- The total memory is obviously `10,000 * 100 = 1,000,000` units - $O(n \times m)$. Counting just "`k` keys + `n` strings" as if they were single units **misses** `m` length per each string.

Store all `n` strings in the output — each one costs $O(m)$ to store, not $O(1)$. So it's $O(n·m)$, not $O(n)$.

> **General rule to keep**: whenever you store or process `n` items and each one individually costs $O(m)$, the total is `n·m`, not `n+m` or just `n`.


In [ ]:
from collections import defaultdict


def group_anagrams(strs: list[str]) -> list[list[str]]:
    result = defaultdict(list)

    for string in strs:
        count = [0] * 26  # a... z

        for char in string:
            # map each char to index 0... 25, and count each one in the string
            count[ord(char) - ord("a")] += 1
            # {"a"(97) - "a"(97) = 0: 1}
            # {"c"(99) - "a"(97) = 2: 1}
            # {"t"(116) - "a"(97) = 19: 1}

        # use tuple as key
        result[tuple(count)].append(string)

    return list(result.values())

In [ ]:
print(group_anagrams(["act","pots","tops","cat","stop","hat"]))
# [['act', 'cat'], ['pots', 'tops', 'stop'], ['hat']]